# Methods Notebook 1: Event Detection

**BRAVOSEIS Hydroacoustic Analysis Pipeline — Step 1**

This notebook walks through the STA/LTA event detection process applied to the BRAVOSEIS hydrophone array data. It demonstrates each stage of the detection pipeline on a single example file, from raw DAT binary ingestion through to a catalogue of characterized events.

**Environment:** Install dependencies from `environment.yml` in this directory, or use the project's existing virtual environment (`uv run`).

**Data requirement:** Access to the raw NOAA/PMEL DAT files at the configured data root path. If raw data is unavailable, the notebook can load pre-computed outputs from `outputs/data/`.

## 0. Environment Setup

**Inputs:** None.

**Outputs:** Loaded libraries, project paths configured, `scripts/` directory added to Python path so that project modules (`read_dat`, `detect_events`) can be imported.

**Adjustable parameters:** 
- `DATA_ROOT` — path to the directory containing mooring subdirectories with `.DAT` files. Change this if your data is stored elsewhere.
- `REPO_ROOT` — path to the repository root. Auto-detected from the notebook location.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfilt, spectrogram

# Project paths
REPO_ROOT = Path.cwd().parent.parent  # notebooks/methods_notebooks -> repo root
SCRIPTS_DIR = REPO_ROOT / "scripts"
OUTPUT_DIR = REPO_ROOT / "outputs" / "data"
DATA_ROOT = Path("/home/jovyan/my_data/bravoseis/NOAA")

# Add scripts to path for project module imports
sys.path.insert(0, str(SCRIPTS_DIR))

from read_dat import read_dat, read_header, list_mooring_files, MOORINGS, SAMPLE_RATE

print(f"Repository root: {REPO_ROOT}")
print(f"Data root:       {DATA_ROOT}")
print(f"Sample rate:     {SAMPLE_RATE} Hz")
print(f"Moorings:        {sorted(MOORINGS.keys())}")

%matplotlib inline
plt.rcParams.update({"figure.figsize": (14, 5), "font.size": 11})

## 1. Load a Raw DAT File

**Inputs:** A single `.DAT` binary file from one mooring. The file contains a 256-byte big-endian header followed by 14,400,000 unsigned 16-bit samples (4 hours at 1 kHz).

**Outputs:**
- `timestamp` — recording start time (UTC), parsed from the header
- `data` — signed acoustic waveform as a 1-D numpy array (float64, centered on zero)
- `metadata` — dict of parsed header fields (sample rate, ADC bytes, duty cycle, instrument ID, etc.)

**Adjustable parameters:**
- `MOORING` — which mooring to load (m1–m6)
- `FILE_INDEX` — which file to load from the mooring's sorted file list (0-indexed). File 50 is roughly mid-deployment.

**Note:** The `read_dat()` function in `scripts/read_dat.py` handles the binary format. The raw data is unsigned 16-bit big-endian; the reader subtracts 32768 to center the waveform on zero. The original ADC is 24-bit but only the top 16 bits are stored.

In [ ]:
# --- Adjustable parameters ---
MOORING = "m1"
FILE_INDEX = 50  # Index into sorted file list; 50 is mid-deployment

# --- Load the file ---
mooring_info = MOORINGS[MOORING]
mooring_dir = DATA_ROOT / mooring_info["data_dir"]
dat_files = sorted(mooring_dir.glob("*.DAT"))
print(f"Mooring {MOORING}: {len(dat_files)} DAT files in {mooring_dir.name}")

dat_file = dat_files[FILE_INDEX]
print(f"Loading file {FILE_INDEX}: {dat_file.name}")

timestamp, data, metadata = read_dat(dat_file)

print(f"\nTimestamp (UTC): {timestamp}")
print(f"Waveform shape:  {data.shape} ({data.shape[0] / SAMPLE_RATE / 3600:.1f} hours)")
print(f"Value range:     [{data.min():.0f}, {data.max():.0f}]")
print(f"\nHeader metadata:")
for k, v in metadata.items():
    print(f"  {k}: {v}")

## 2. Visualize the Raw Waveform

**Inputs:** The `data` array and `timestamp` from the previous cell.

**Outputs:** A time-domain plot of the full 4-hour waveform, showing the raw acoustic amplitude over time.

**Purpose:** Visual inspection of the recording. Transient events (earthquakes, icequakes) appear as amplitude spikes above the ambient noise floor. Vessel passages appear as sustained amplitude increases lasting hours.

In [ ]:
time_hours = np.arange(len(data)) / SAMPLE_RATE / 3600

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(time_hours, data, linewidth=0.1, color="k", rasterized=True)
ax.set_xlabel("Time (hours from file start)")
ax.set_ylabel("Amplitude (counts)")
ax.set_title(f"Raw Waveform — {MOORING.upper()}, {timestamp.strftime('%Y-%m-%d %H:%M UTC')}")
ax.set_xlim(0, time_hours[-1])
plt.tight_layout()
plt.show()

## 3. Bandpass Filtering — Three Frequency Passes

**Inputs:** The raw `data` waveform (1-D array, 14.4M samples).

**Outputs:** Three filtered waveforms, one per frequency band:
- `filtered_low` — 1–15 Hz (targets: T-phases, earthquakes)
- `filtered_mid` — 15–30 Hz (targets: fin whale calls at ~20 Hz, icequakes)
- `filtered_high` — 30–250 Hz (targets: icequakes, vessel noise, whale calls)

**Adjustable parameters:**
- `BAND_BREAKPOINTS` — the frequency boundaries separating the three passes. Currently set to 15 Hz and 30 Hz. These were chosen from average spectral profiles: dominant seismic/T-phase energy falls below 15 Hz, and fin whale 20 Hz calls occupy the 15–30 Hz band.
- `FILTER_ORDER` — Butterworth filter order (default 4). Higher orders give sharper rolloff but may introduce ringing.

**Why three passes?** A single broadband STA/LTA detector allows high-energy low-frequency events (T-phases) to dominate the LTA window, suppressing detection of concurrent higher-frequency signals. Separating into non-overlapping bands ensures each pass's LTA reflects only its own frequency regime. This recovered 7x more mid-band events compared to the initial overlapping-band approach.

In [ ]:
# --- Adjustable parameters ---
BAND_BREAKPOINTS = (15, 30)  # Hz — boundaries between low/mid and mid/high
FILTER_ORDER = 4             # Butterworth filter order

# --- Define the three frequency bands ---
BANDS = {
    "low":  {"range": (1, BAND_BREAKPOINTS[0]),   "color": "#d95f02"},
    "mid":  {"range": (BAND_BREAKPOINTS[0], BAND_BREAKPOINTS[1]), "color": "#1b9e77"},
    "high": {"range": (BAND_BREAKPOINTS[1], 250), "color": "#7570b3"},
}

def apply_bandpass(data, low_hz, high_hz, fs=SAMPLE_RATE, order=FILTER_ORDER):
    """Apply a Butterworth bandpass filter."""
    nyq = fs / 2
    low = max(low_hz / nyq, 0.001)
    high = min(high_hz / nyq, 0.999)
    sos = butter(order, [low, high], btype="bandpass", output="sos")
    return sosfilt(sos, data)

# --- Apply filters ---
filtered = {}
for band_name, band_cfg in BANDS.items():
    lo, hi = band_cfg["range"]
    filtered[band_name] = apply_bandpass(data, lo, hi)
    print(f"  {band_name:5s} ({lo:3d}–{hi:3d} Hz): RMS = {np.sqrt(np.mean(filtered[band_name]**2)):.1f}")

# --- Plot filtered waveforms (first 10 minutes for detail) ---
show_minutes = 10
show_samples = int(show_minutes * 60 * SAMPLE_RATE)
time_sec = np.arange(show_samples) / SAMPLE_RATE

fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
for ax, (band_name, band_cfg) in zip(axes, BANDS.items()):
    lo, hi = band_cfg["range"]
    ax.plot(time_sec, filtered[band_name][:show_samples],
            linewidth=0.3, color=band_cfg["color"], rasterized=True)
    ax.set_ylabel(f"{band_name}\n({lo}–{hi} Hz)")
    ax.set_xlim(0, time_sec[-1])

axes[-1].set_xlabel("Time (seconds from file start)")
axes[0].set_title(f"Bandpass-Filtered Waveforms — First {show_minutes} Minutes")
plt.tight_layout()
plt.show()

## 4. STA/LTA Computation

**Inputs:** One filtered waveform (1-D array). This cell demonstrates the STA/LTA computation on the low-band (1–15 Hz) waveform.

**Outputs:**
- `envelope` — squared amplitude of the filtered signal (energy proxy)
- `cft` — the STA/LTA characteristic function, a 1-D array the same length as the input. Values above the trigger threshold indicate detected events.

**Adjustable parameters:**
- `STA_SEC` (2.0 s) — short-term average window length. Controls sensitivity to impulsive onsets. Shorter values (e.g., 0.25 s as in Wilcock 2012) are better for brief pulses like fin whale calls. Longer values (2.0 s) average over noise fluctuations and are appropriate for T-phases with onsets < 1 s.
- `LTA_SEC` (60.0 s) — long-term average window length. Represents the ambient noise floor. Must be long enough to span several event durations so that individual events do not bias the background estimate. 60 s is standard across Wilcock (2012), Soule & Wilcock (2013), and this work.
- `TRIGGER` (3.0) — STA/LTA ratio above which an event is declared. Standard value for regional seismo-acoustic networks (Fox et al. 2001). Lower values increase sensitivity but also increase false triggers.
- `DETRIGGER` (1.5) — STA/LTA ratio below which an ongoing event is declared ended. Typically set to half the trigger threshold.

**Algorithm:** The `classic_sta_lta()` function computes the ratio using numpy cumulative sums for O(n) performance. The energy envelope (squared amplitude) is used rather than raw amplitude to emphasize transient energy bursts.

In [ ]:
# --- Adjustable parameters ---
STA_SEC = 2.0       # Short-term average window (seconds)
LTA_SEC = 60.0      # Long-term average window (seconds)
TRIGGER = 3.0       # STA/LTA trigger threshold
DETRIGGER = 1.5     # STA/LTA detrigger threshold

# Convert to samples
NSTA = int(STA_SEC * SAMPLE_RATE)
NLTA = int(LTA_SEC * SAMPLE_RATE)

print(f"STA: {STA_SEC}s = {NSTA} samples")
print(f"LTA: {LTA_SEC}s = {NLTA} samples")

def classic_sta_lta(data, nsta, nlta):
    """Classic STA/LTA using cumulative sum (fully vectorized).
    
    Computes the ratio of short-term to long-term average energy.
    The output is valid from index nlta onward; earlier values are zero.
    """
    ndat = len(data)
    cs = np.zeros(ndat + 1, dtype=np.float64)
    np.cumsum(data, out=cs[1:])
    
    sta = np.zeros(ndat, dtype=np.float64)
    lta = np.zeros(ndat, dtype=np.float64)
    
    sta[nsta:] = (cs[nsta + 1:] - cs[1:-nsta]) / nsta
    lta[nlta:] = (cs[nlta + 1:] - cs[1:-nlta]) / nlta
    
    cft = np.zeros(ndat, dtype=np.float64)
    valid = lta > 0
    cft[valid] = sta[valid] / lta[valid]
    return cft

# --- Compute on low band ---
DEMO_BAND = "low"
envelope = filtered[DEMO_BAND] ** 2
cft = classic_sta_lta(envelope, NSTA, NLTA)

print(f"\nSTA/LTA computed on {DEMO_BAND} band")
print(f"  Max STA/LTA ratio: {cft.max():.1f}")
print(f"  Samples above trigger ({TRIGGER}): {(cft >= TRIGGER).sum():,}")

# --- Plot STA/LTA function for first 30 minutes ---
show_sec = 30 * 60
show_samp = int(show_sec * SAMPLE_RATE)
t = np.arange(show_samp) / SAMPLE_RATE

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

ax1.plot(t, filtered[DEMO_BAND][:show_samp], linewidth=0.3, color="k", rasterized=True)
ax1.set_ylabel("Amplitude")
ax1.set_title(f"Filtered Waveform and STA/LTA — {DEMO_BAND} band ({BANDS[DEMO_BAND]['range'][0]}–{BANDS[DEMO_BAND]['range'][1]} Hz)")

ax2.plot(t, cft[:show_samp], linewidth=0.5, color="steelblue")
ax2.axhline(TRIGGER, color="red", linestyle="--", linewidth=1, label=f"Trigger = {TRIGGER}")
ax2.axhline(DETRIGGER, color="orange", linestyle="--", linewidth=1, label=f"Detrigger = {DETRIGGER}")
ax2.set_ylabel("STA/LTA Ratio")
ax2.set_xlabel("Time (seconds from file start)")
ax2.legend(loc="upper right")
ax2.set_ylim(0, min(cft[:show_samp].max() * 1.2, 50))

plt.tight_layout()
plt.show()

## 5. Trigger/Detrigger — Extracting Event Windows

**Inputs:** The `cft` (STA/LTA characteristic function) array from the previous cell.

**Outputs:**
- `on_off` — an Nx2 array of [trigger_sample, detrigger_sample] pairs, one row per raw detection
- `events` — filtered list of event dicts after applying minimum duration and minimum gap constraints

**Adjustable parameters:**
- `MIN_DURATION` (0.5 s) — events shorter than this are discarded. Rejects ADC artifacts and single-sample spikes that briefly exceed the trigger threshold.
- `MIN_GAP` (2.0 s) — if two events are separated by less than this, they are merged into one. Prevents fragmentation of long events that have brief amplitude dips below the detrigger threshold mid-event.

**Algorithm:** The trigger/detrigger logic walks through the STA/LTA function sample by sample. An event begins when the ratio exceeds `TRIGGER` and ends when it drops below `DETRIGGER`. After extracting raw trigger pairs, two post-processing filters are applied:
1. Events shorter than `MIN_DURATION` are discarded
2. Events separated by less than `MIN_GAP` are merged (the earlier event's end is extended to the later event's end)

In [ ]:
# --- Adjustable parameters ---
MIN_DURATION = 0.5  # Minimum event duration (seconds)
MIN_GAP = 2.0       # Minimum inter-event gap (seconds)

MIN_DUR_SAMP = int(MIN_DURATION * SAMPLE_RATE)
MIN_GAP_SAMP = int(MIN_GAP * SAMPLE_RATE)

def trigger_onset(cft, threshold_on, threshold_off):
    """Extract trigger on/off pairs from STA/LTA function."""
    on = False
    triggers = []
    on_idx = 0
    for i in range(len(cft)):
        if not on and cft[i] >= threshold_on:
            on = True
            on_idx = i
        elif on and cft[i] <= threshold_off:
            on = False
            triggers.append([on_idx, i])
    if on:
        triggers.append([on_idx, len(cft) - 1])
    return np.array(triggers, dtype=np.int64) if triggers else np.empty((0, 2), dtype=np.int64)

# --- Extract raw triggers ---
on_off = trigger_onset(cft, TRIGGER, DETRIGGER)
print(f"Raw trigger pairs: {len(on_off)}")

# --- Apply duration and gap filters ---
events = []
prev_off = -MIN_GAP_SAMP - 1

for on_samp, off_samp in on_off:
    duration_samp = off_samp - on_samp
    if duration_samp < MIN_DUR_SAMP:
        continue
    if on_samp - prev_off < MIN_GAP_SAMP and events:
        # Merge with previous event
        events[-1]["off_samp"] = off_samp
        prev_off = off_samp
        continue
    events.append({"on_samp": int(on_samp), "off_samp": int(off_samp)})
    prev_off = off_samp

n_rejected_dur = sum(1 for on, off in on_off if (off - on) < MIN_DUR_SAMP)
n_merged = len(on_off) - n_rejected_dur - len(events)

print(f"After filtering:")
print(f"  Rejected (< {MIN_DURATION}s): {n_rejected_dur}")
print(f"  Merged (gap < {MIN_GAP}s):    {n_merged}")
print(f"  Final events:                 {len(events)}")

# --- Show first few events ---
print(f"\nFirst 5 events:")
for i, ev in enumerate(events[:5]):
    dur = (ev["off_samp"] - ev["on_samp"]) / SAMPLE_RATE
    t_on = ev["on_samp"] / SAMPLE_RATE
    print(f"  Event {i}: onset={t_on:.2f}s, duration={dur:.2f}s")

## 6. Event Characterization — Spectral Features

**Inputs:** The raw (unfiltered) `data` waveform and the `events` list from the previous cell. Each event is defined by its on/off sample indices.

**Outputs:** For each event, the following features are computed and stored in a list of dicts:
- `onset_utc` — absolute UTC time of the event onset
- `duration_s` — event duration in seconds (trigger to detrigger)
- `peak_freq_hz` — frequency with maximum mean power in the event spectrogram
- `bandwidth_hz` — frequency range containing 90% of the event energy (5th to 95th percentile of cumulative power)
- `peak_db` — maximum power in the event spectrogram (dB, relative)
- `snr` — peak STA/LTA ratio during the event window

**Adjustable parameters:**
- `NPERSEG` (1024) — FFT window length for the spectrogram, in samples. At 1 kHz, this gives ~1 s windows with ~1 Hz frequency resolution. Shorter windows give better time resolution but coarser frequency resolution.
- `NOVERLAP` (512) — overlap between successive FFT windows, in samples. 50% overlap is standard.
- `FREQ_MAX` (250 Hz) — maximum frequency to retain from the spectrogram. The Nyquist frequency is 500 Hz but there is negligible signal energy above 250 Hz.
- `PAD_SEC` (2) — seconds of context padding added before and after each event for spectrogram computation. This ensures the FFT windows at event boundaries have sufficient context.

**Algorithm:** A spectrogram (scipy.signal.spectrogram) is computed on a padded segment of the raw waveform around each event. Spectral features are extracted from the spectrogram columns that fall within the event window (excluding the padding). Power is computed in dB (10*log10).

In [ ]:
# --- Adjustable parameters ---
NPERSEG = 1024   # FFT window length (samples); ~1s at 1 kHz
NOVERLAP = 512   # FFT overlap (samples); 50%
FREQ_MAX = 250   # Max frequency to retain (Hz)
PAD_SEC = 2      # Context padding around each event (seconds)

from datetime import timedelta

pad_samp = PAD_SEC * SAMPLE_RATE

detections = []
for ev in events:
    on_s = ev["on_samp"]
    off_s = ev["off_samp"]

    # Pad for spectrogram context
    seg_start = max(0, on_s - pad_samp)
    seg_end = min(len(data), off_s + pad_samp)
    segment = data[seg_start:seg_end]

    if len(segment) < NPERSEG:
        continue

    # Compute spectrogram
    freqs, times, Sxx = spectrogram(
        segment, fs=SAMPLE_RATE, nperseg=NPERSEG, noverlap=NOVERLAP
    )

    # Clip to frequency range
    freq_mask = freqs <= FREQ_MAX
    freqs = freqs[freq_mask]
    Sxx = Sxx[freq_mask, :]

    if Sxx.size == 0:
        continue

    Sxx_dB = 10 * np.log10(Sxx + 1e-20)

    # Isolate the event region within the padded spectrogram
    event_t_start = (on_s - seg_start) / SAMPLE_RATE
    event_t_end = (off_s - seg_start) / SAMPLE_RATE
    t_mask = (times >= event_t_start) & (times <= event_t_end)

    if not np.any(t_mask):
        t_mask = np.ones(len(times), dtype=bool)

    event_Sxx = Sxx[:, t_mask]
    event_dB = Sxx_dB[:, t_mask]

    # Peak frequency
    mean_power = event_Sxx.mean(axis=1)
    peak_freq_hz = float(freqs[np.argmax(mean_power)])

    # Peak power (dB)
    peak_db = float(event_dB.max())

    # Bandwidth (90% energy)
    total_power = mean_power.sum()
    if total_power > 0:
        cumsum = np.cumsum(mean_power)
        frac = cumsum / total_power
        low_idx = np.searchsorted(frac, 0.05)
        high_idx = np.searchsorted(frac, 0.95)
        bandwidth_hz = float(freqs[min(high_idx, len(freqs)-1)]
                             - freqs[max(low_idx, 0)])
    else:
        bandwidth_hz = 0.0

    # SNR
    cft_event = cft[on_s:off_s]
    snr = float(cft_event.max()) if len(cft_event) > 0 else 0.0

    # Timing
    onset_sec = on_s / SAMPLE_RATE
    duration_sec = (off_s - on_s) / SAMPLE_RATE
    onset_utc = timestamp + timedelta(seconds=onset_sec)

    detections.append({
        "onset_utc": onset_utc,
        "duration_s": round(duration_sec, 3),
        "peak_freq_hz": round(peak_freq_hz, 1),
        "bandwidth_hz": round(bandwidth_hz, 1),
        "peak_db": round(peak_db, 1),
        "snr": round(snr, 2),
        "detection_band": DEMO_BAND,
        "on_samp": on_s,
        "off_samp": off_s,
    })

det_df = pd.DataFrame(detections)
print(f"Characterized {len(det_df)} events in the {DEMO_BAND} band")
print(f"\nFeature summary:")
print(det_df[["duration_s", "peak_freq_hz", "bandwidth_hz", "peak_db", "snr"]].describe().round(1))

## 7. Visualize Detected Events on Spectrogram

**Inputs:** The raw `data` waveform and the `detections` list from the previous cell.

**Outputs:** A spectrogram of a selected time window with detected event boundaries overlaid as vertical red lines. This provides visual confirmation that the STA/LTA detector is triggering on real acoustic events and that the event boundaries are reasonable.

**Adjustable parameters:**
- `PLOT_START_SEC` / `PLOT_DURATION_SEC` — time window to display. Adjust these to zoom into regions of interest (e.g., a swarm, a quiet period, or a vessel passage).

**What to look for:**
- Red lines should bracket visible transient events in the spectrogram
- Events missed by the detector (visible transients without red brackets) suggest the trigger threshold may be too high
- False triggers (red brackets over noise with no visible transient) suggest the threshold may be too low

In [ ]:
# --- Adjustable parameters ---
PLOT_START_SEC = 0       # Start of display window (seconds from file start)
PLOT_DURATION_SEC = 600  # Duration of display window (seconds); 600 = 10 min

# --- Extract segment ---
s0 = int(PLOT_START_SEC * SAMPLE_RATE)
s1 = int((PLOT_START_SEC + PLOT_DURATION_SEC) * SAMPLE_RATE)
segment = data[s0:s1]

# Compute spectrogram for display
freqs_sg, times_sg, Sxx_sg = spectrogram(
    segment, fs=SAMPLE_RATE, nperseg=NPERSEG, noverlap=NOVERLAP
)
freq_mask = freqs_sg <= FREQ_MAX
Sxx_dB_sg = 10 * np.log10(Sxx_sg[freq_mask, :] + 1e-20)

# Percentile color limits
vmin, vmax = np.percentile(Sxx_dB_sg, [2, 98])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                                gridspec_kw={"height_ratios": [3, 1]})

# Spectrogram
ax1.pcolormesh(times_sg + PLOT_START_SEC, freqs_sg[freq_mask], Sxx_dB_sg,
               shading="auto", cmap="viridis", vmin=vmin, vmax=vmax, rasterized=True)
ax1.set_ylabel("Frequency (Hz)")
ax1.set_title(f"Spectrogram with Detected Events — {DEMO_BAND} band")

# Overlay event boundaries
for det in detections:
    t_on = det["on_samp"] / SAMPLE_RATE
    t_off = det["off_samp"] / SAMPLE_RATE
    if PLOT_START_SEC <= t_on <= PLOT_START_SEC + PLOT_DURATION_SEC:
        ax1.axvline(t_on, color="red", linewidth=0.8, alpha=0.7)
        ax1.axvline(t_off, color="red", linewidth=0.8, alpha=0.7, linestyle="--")

# STA/LTA in same window
t_cft = np.arange(s0, s1) / SAMPLE_RATE
ax2.plot(t_cft, cft[s0:s1], linewidth=0.5, color="steelblue")
ax2.axhline(TRIGGER, color="red", linestyle="--", linewidth=1, label=f"Trigger = {TRIGGER}")
ax2.axhline(DETRIGGER, color="orange", linestyle="--", linewidth=1, label=f"Detrigger = {DETRIGGER}")
ax2.set_ylabel("STA/LTA")
ax2.set_xlabel("Time (seconds from file start)")
ax2.set_ylim(0, min(cft[s0:s1].max() * 1.2, 30))
ax2.legend(loc="upper right", fontsize=9)

plt.tight_layout()
plt.show()

## 8. Run All Three Bands — Full File Detection

**Inputs:** The raw `data` waveform and the three band definitions from Section 3.

**Outputs:** A combined DataFrame (`all_detections`) containing all events detected across all three frequency bands for this single file. Each row includes the band label and all spectral features.

**Note:** In the production pipeline (`scripts/detect_events.py`), this process runs in parallel across all files and all moorings. Here we demonstrate it on a single file. Events detected in multiple bands are not deduplicated at this stage — that is handled during downstream classification (Phase 3 frequency-band separation).

In [ ]:
def detect_one_band(data, band_name, band_range, file_timestamp):
    """Run the full detection pipeline on one frequency band.
    
    Returns a list of detection dicts.
    """
    lo, hi = band_range
    filt = apply_bandpass(data, lo, hi)
    env = filt ** 2
    cft_band = classic_sta_lta(env, NSTA, NLTA)
    on_off_band = trigger_onset(cft_band, TRIGGER, DETRIGGER)

    if len(on_off_band) == 0:
        return []

    # Duration + gap filtering
    evts = []
    prev_off = -MIN_GAP_SAMP - 1
    for on_samp, off_samp in on_off_band:
        if (off_samp - on_samp) < MIN_DUR_SAMP:
            continue
        if on_samp - prev_off < MIN_GAP_SAMP and evts:
            evts[-1]["off_samp"] = off_samp
            prev_off = off_samp
            continue
        evts.append({"on_samp": int(on_samp), "off_samp": int(off_samp)})
        prev_off = off_samp

    # Characterize
    dets = []
    pad = PAD_SEC * SAMPLE_RATE
    for ev in evts:
        on_s, off_s = ev["on_samp"], ev["off_samp"]
        seg_start = max(0, on_s - pad)
        seg_end = min(len(data), off_s + pad)
        segment = data[seg_start:seg_end]
        if len(segment) < NPERSEG:
            continue

        freqs_c, times_c, Sxx_c = spectrogram(
            segment, fs=SAMPLE_RATE, nperseg=NPERSEG, noverlap=NOVERLAP
        )
        fm = freqs_c <= FREQ_MAX
        freqs_c, Sxx_c = freqs_c[fm], Sxx_c[fm, :]
        if Sxx_c.size == 0:
            continue

        et_start = (on_s - seg_start) / SAMPLE_RATE
        et_end = (off_s - seg_start) / SAMPLE_RATE
        tm = (times_c >= et_start) & (times_c <= et_end)
        if not np.any(tm):
            tm = np.ones(len(times_c), dtype=bool)

        mp = Sxx_c[:, tm].mean(axis=1)
        tp = mp.sum()
        if tp > 0:
            cs = np.cumsum(mp) / tp
            bw = float(freqs_c[min(np.searchsorted(cs, 0.95), len(freqs_c)-1)]
                       - freqs_c[max(np.searchsorted(cs, 0.05), 0)])
        else:
            bw = 0.0

        cft_ev = cft_band[on_s:off_s]
        onset_utc = file_timestamp + timedelta(seconds=on_s / SAMPLE_RATE)

        dets.append({
            "onset_utc": onset_utc,
            "duration_s": round((off_s - on_s) / SAMPLE_RATE, 3),
            "peak_freq_hz": round(float(freqs_c[np.argmax(mp)]), 1),
            "bandwidth_hz": round(bw, 1),
            "peak_db": round(float(10 * np.log10(Sxx_c[:, tm].max() + 1e-20)), 1),
            "snr": round(float(cft_ev.max()) if len(cft_ev) > 0 else 0.0, 2),
            "detection_band": band_name,
        })
    return dets

# --- Run all three bands ---
all_dets = []
for band_name, band_cfg in BANDS.items():
    dets = detect_one_band(data, band_name, band_cfg["range"], timestamp)
    all_dets.extend(dets)
    print(f"  {band_name:5s}: {len(dets):5d} events")

all_detections = pd.DataFrame(all_dets)
print(f"\nTotal: {len(all_detections)} events in this file")
print(f"\nBy band:")
print(all_detections["detection_band"].value_counts().to_string())

## 9. Duration vs. Peak Frequency — Feature Space Validation

**Inputs:** The `all_detections` DataFrame from the previous cell.

**Outputs:** A scatter plot of event duration versus peak frequency, colored by detection band. This plot validates that the three-band strategy captures distinct source populations.

**What to look for:**
- Low-band events should cluster at low frequencies (1–15 Hz) with a range of durations
- Mid-band events should occupy 15–30 Hz
- High-band events should appear above 30 Hz
- If populations overlap heavily, the band boundaries may need adjustment

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for band_name, band_cfg in BANDS.items():
    mask = all_detections["detection_band"] == band_name
    subset = all_detections[mask]
    lo, hi = band_cfg["range"]
    ax.scatter(subset["peak_freq_hz"], subset["duration_s"],
               s=8, alpha=0.5, color=band_cfg["color"],
               label=f"{band_name} ({lo}–{hi} Hz): {len(subset)} events")

ax.set_xlabel("Peak Frequency (Hz)")
ax.set_ylabel("Duration (s)")
ax.set_yscale("log")
ax.set_title("Duration vs. Peak Frequency — Single File Detection")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Compare with Production Catalogue

**Inputs:**
- `all_detections` — events detected in this notebook from the single demonstration file
- `outputs/data/event_catalogue.parquet` — the full production catalogue (297,170 events from 717 files)

**Outputs:** A comparison of event counts and feature distributions between the notebook's single-file output and the production catalogue, filtered to the same mooring and file. This verifies that the notebook reproduces the production pipeline.

**Note:** Small differences may occur due to floating-point precision or edge effects at file boundaries. The counts should match exactly if the parameters are identical.

In [ ]:
# Load the production catalogue
cat_path = OUTPUT_DIR / "event_catalogue.parquet"
if cat_path.exists():
    cat = pd.read_parquet(cat_path)
    cat["onset_utc"] = pd.to_datetime(cat["onset_utc"])

    # Filter to this mooring and this file's time window
    file_start = pd.Timestamp(timestamp)
    file_end = file_start + pd.Timedelta(hours=4)

    cat_file = cat[(cat["mooring"] == MOORING) &
                   (cat["onset_utc"] >= file_start) &
                   (cat["onset_utc"] < file_end)]

    print(f"Production catalogue events for {MOORING}, {timestamp.date()}:")
    print(f"  Total: {len(cat_file)}")
    print(f"  By band:")
    print(cat_file["detection_band"].value_counts().to_string())
    print(f"\nNotebook detections:")
    print(f"  Total: {len(all_detections)}")
    print(f"  By band:")
    print(all_detections["detection_band"].value_counts().to_string())

    # Compare feature distributions
    if len(cat_file) > 0 and len(all_detections) > 0:
        print(f"\nFeature comparison (median values):")
        print(f"{'Feature':<16s} {'Production':>12s} {'Notebook':>12s}")
        print(f"{'-'*16} {'-'*12} {'-'*12}")
        for col in ["duration_s", "peak_freq_hz", "snr"]:
            if col in cat_file.columns and col in all_detections.columns:
                prod_med = cat_file[col].median()
                nb_med = all_detections[col].median()
                print(f"{col:<16s} {prod_med:>12.2f} {nb_med:>12.2f}")
else:
    print(f"Production catalogue not found at {cat_path}")
    print("This is expected if running outside the project environment.")

## 11. Summary of Adjustable Parameters

All parameters that control the detection pipeline are collected here for reference. Changing any of these values and re-running the notebook from the top will produce different detection results.

| Parameter | Current Value | Cell | Effect of Increasing | Effect of Decreasing |
|-----------|--------------|------|---------------------|---------------------|
| `DATA_ROOT` | `/home/jovyan/my_data/bravoseis/NOAA` | 0 | — | — |
| `MOORING` | `m1` | 1 | — | — |
| `FILE_INDEX` | `50` | 1 | Later in deployment | Earlier in deployment |
| `BAND_BREAKPOINTS` | `(15, 30)` Hz | 3 | More events in low band, fewer in mid | More events in mid band, fewer in low |
| `FILTER_ORDER` | `4` | 3 | Sharper filter rolloff, possible ringing | Gentler rolloff, more spectral leakage |
| `STA_SEC` | `2.0` s | 4 | Smoother STA, may miss brief events | More sensitive to short transients |
| `LTA_SEC` | `60.0` s | 4 | More stable background estimate | Background tracks signal more closely |
| `TRIGGER` | `3.0` | 4 | Fewer events (higher SNR required) | More events (more false positives) |
| `DETRIGGER` | `1.5` | 4 | Shorter events (ends sooner) | Longer events (includes more coda) |
| `MIN_DURATION` | `0.5` s | 5 | Rejects more short events | Accepts more glitches |
| `MIN_GAP` | `2.0` s | 5 | More merging of adjacent events | More fragmented events |
| `NPERSEG` | `1024` samples | 6 | Better freq resolution, worse time resolution | Better time resolution, worse freq resolution |
| `FREQ_MAX` | `250` Hz | 6 | Includes more high-freq content | Excludes high-freq content |

**Production script:** `scripts/detect_events.py` applies this same pipeline to all files across all moorings with multiprocessing parallelism. The full 14,663-file archive produces 6,761,070 events.